<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.6-observability-with-langfuse/notebooks/exercises-5.6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.6 — Observability & Langfuse
Netsetos GenAI Course v2.0 | Module 5

Tracing, evaluation, monitoring, prompt management, India compliance


In [ ]:
# pip install langfuse openai ragas deepeval -q
print('Ready for LLM observability')


## Ex 1: Langfuse @observe Tracing


In [ ]:
from langfuse import observe, get_client
from openai import OpenAI

langfuse = get_client()
client = OpenAI()

@observe(name='chat-pipeline')
def chat(user_msg):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role':'user','content':user_msg}]
    )
    return response.choices[0].message.content

# result = chat('What is observability?')
print('Langfuse @observe: auto-traces LLM calls as nested spans')
print('View at: cloud.langfuse.com/traces')


## Ex 2: Helicone 1-Line Integration


In [ ]:
from openai import OpenAI

# Change ONE line for full observability
client = OpenAI(
    base_url='https://oai.helicone.ai/v1',
    default_headers={
        'Helicone-Auth': 'Bearer YOUR_KEY',
        'Helicone-Property-Feature': 'chatbot',
        'Helicone-Property-Environment': 'dev'
    }
)

for i in range(5):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role':'user','content':f'Q{i}'}], max_tokens=50)
    print(f'Req {i}: {r.usage.total_tokens} tokens')
print('Dashboard: helicone.ai/dashboard')


## Ex 3: Prompt Management


In [ ]:
# Langfuse prompt versioning
from langfuse import get_client
langfuse = get_client()

# Fetch production prompt (cached client-side)
# prompt = langfuse.get_prompt('movie-critic', label='production')
# compiled = prompt.compile(movie='Dune 2', criticlevel='expert')
# model = prompt.config.get('model', 'gpt-4o')

print('Prompt Management Features:')
for f in ['Versioned (immutable, auto-incrementing)',
          'Labeled (production/staging/latest)',
          'Templated ({{variable}} syntax)',
          'Cached (client-side, configurable TTL)',
          'A/B testable (experiments with LLM-as-judge)',
          'Rollback (reassign production label)']:
    print(f'  ✅ {f}')


## Ex 4: RAGAS Evaluation


In [ ]:
# RAGAS faithfulness evaluation
# from ragas.metrics import Faithfulness
# scorer = Faithfulness(llm=llm)
# result = await scorer.ascore(
#     user_input='When was the first super bowl?',
#     response='The first superbowl was on Jan 15, 1967',
#     retrieved_contexts=['The First AFL-NFL Championship was Jan 15, 1967...']
# )

print('RAGAS Core Metrics (0-1):')
metrics = [
    ('Faithfulness', 'Claims supported by context (hallucination detection)'),
    ('Response Relevancy', 'Answer relevance to question (embedding similarity)'),
    ('Context Precision', 'Retrieved docs relevance (precision@k)'),
    ('Context Recall', 'Coverage of relevant info (needs ground truth)'),
]
for name, desc in metrics:
    print(f'  {name:25s}: {desc}')


## Ex 5: LLM-as-Judge Pattern


In [ ]:
from openai import OpenAI
client = OpenAI()

def llm_judge(question, answer, criteria='relevance'):
    prompt = f'''Rate the following answer on {criteria} (1-5).
    
Question: {question}
Answer: {answer}

First explain your reasoning, then give a score.
Format: Reasoning: ... Score: N'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role':'user','content':prompt}],
        max_tokens=200
    )
    return resp.choices[0].message.content

print('LLM-as-Judge Best Practices:')
for p in ['Rationale BEFORE score (not after)',
          'Binary yes/no over multi-point scales',
          'Separate judge from generator model',
          'Randomize option order (position bias)',
          'Multi-agent ensemble (majority vote)']:
    print(f'  ✅ {p}')


## Ex 6: OpenTelemetry + OpenLLMetry


In [ ]:
# OpenLLMetry: vendor-neutral OTEL instrumentation
# from traceloop.sdk import Traceloop
# Traceloop.init()  # Auto-instruments 20+ providers

print('OpenTelemetry GenAI Semantic Conventions:')
attrs = [
    'gen_ai.request.model',
    'gen_ai.usage.input_tokens',
    'gen_ai.usage.output_tokens',
    'gen_ai.operation.name',
    'gen_ai.agent.name',
]
for a in attrs:
    print(f'  {a}')

print('\nExport to any backend simultaneously:')
print('  Traceloop.init() → OTEL Collector → Grafana + Langfuse + Datadog')
print('  Overhead: <1ms per LLM call')


## Ex 7: Production Monitoring Dashboard


In [ ]:
print('Key LLM Production Metrics:')
metrics = [
    ('P50/P95/P99 latency', 'Response time distribution'),
    ('TTFB', 'Time to first token (streaming)'),
    ('Tokens/request', 'Prompt + completion token usage'),
    ('Cost/request', 'Per-request cost in USD/INR'),
    ('Cost/user', 'Attribution by user_id'),
    ('Cost/feature', 'Attribution by feature tag'),
    ('Error rate', 'API failures, timeouts, rate limits'),
    ('Quality scores', 'Faithfulness, relevancy (sampled)'),
]
for name, desc in metrics:
    print(f'  {name:20s} → {desc}')

print('\nAlerting (SLO burn rates, not raw thresholds):')
alerts = [
    'Error rate >5% over 15min',
    'P95 latency >2× baseline sustained 5min',
    'Hourly spend >150% of 7-day average',
    'Faithfulness drop >6 points from baseline',
]
for a in alerts:
    print(f'  ⚠️ {a}')


## Ex 8: India DPDP Compliance


In [ ]:
print('DPDP Act Compliance for LLM Observability:')
print(f'  Deadline: May 13, 2027')
print(f'  Penalties: up to ₹250 Cr (~$30M)')
print()

print('✅ CAN log without concern:')
for item in ['Anonymized/aggregated metrics','System metadata','Prompt templates','Model configuration']:
    print(f'    {item}')

print('\n⚠️ MUST handle carefully:')
for item in ['User prompts with PII (Aadhaar, phone, health)','Model outputs with personal data','Session IDs linkable to users']:
    print(f'    {item}')

print('\n🏗️ Setup:')
for item in ['Self-host Langfuse on AWS Mumbai (no India cloud region)','PII redaction: Protecto.ai or @observe(capture_input=False)','INR tracking: USD → ₹ (RBI rates) + 18% GST','Cost: ₹15K-25K/month dev, ₹80K-1.5L/month prod']:
    print(f'    {item}')


---
## Recap
Helicone for instant cost visibility (1-line). Langfuse for deep tracing + prompt management + experiments + annotation queues. RAGAS for RAG evaluation (faithfulness = hallucination detection). DeepEval for pytest-style CI/CD quality gates. OpenTelemetry + OpenLLMetry for vendor-neutral instrumentation. Grafana for dashboards + SLO-based alerting. Model drift detection via embedding comparison + periodic golden dataset evaluation. The evaluation flywheel (online → dataset → offline) is the most impactful pattern. India: self-host Langfuse on AWS Mumbai, PII redaction before trace ingestion, INR + GST cost tracking, ChrF for Indic evaluation. DPDP deadline: May 2027.
